# AI Agent Loop Demo
*Companion notebook for Chapter 25: AI Agents*

This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

We use the **Google Gemini API (free tier)**, which supports function calling and requires no payment to get started. The notebook is self-contained and runs entirely in Colab.

**What you will see:**
- How to define a tool that a model can call
- How the model decides when and how to call it
- How tool results feed back into the next step
- How this loop produces multi-step, goal-directed behavior

The example task: a research assistant that searches a small collection of lab methodology documents to answer questions about protocols and procedures.

## Step 1 — Get a free Gemini API key

You need a Google account. If you are using Colab, you almost certainly already have one.

**How to get the key (takes about two minutes):**

1. Go to [aistudio.google.com](https://aistudio.google.com)
2. Sign in with your Google account
3. Click **"Get API key"** in the left sidebar
4. Click **"Create API key"** — choose "Create API key in new project" if prompted
5. Copy the key that appears (it starts with `AIza...`)

The free tier gives you a generous number of requests per day, which is more than enough for this notebook.

---

**How to store the key securely in Colab (recommended):**

Rather than pasting your key directly into a code cell (where it could be accidentally shared), use Colab's built-in Secrets manager:

1. Click the **🔑 key icon** in the left sidebar of Colab
2. Click **"Add new secret"**
3. Name it exactly: `GOOGLE_API_KEY`
4. Paste your API key as the value
5. Toggle the switch to enable access for this notebook

Once you have done this, the cell below will load the key automatically. You only need to do this setup once — the secret is saved to your Google account and will be available in future Colab sessions.

In [ ]:
import os

# Try to load from Colab Secrets first (recommended)
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab Secrets.")

# Fall back to manual entry if Secrets are not set up
except Exception:
    import getpass
    key = getpass.getpass("Paste your Gemini API key and press Enter: ")
    os.environ["GOOGLE_API_KEY"] = key
    print("API key set.")


The `getpass` fallback above prompts you for the key without showing it on screen, which is safer than typing it directly into a cell.

---

## Step 2 — Install the Gemini SDK

In [ ]:
!pip install google-generativeai --quiet
print("Done.")

## Step 3 — Define the document collection

This is our stand-in for a real research knowledge base. In a real project this would be embedded documents in a vector database (see Chapter 24). Here we use a small set of methodology excerpts to keep things self-contained.

In [ ]:
DOCUMENTS = {
    "irb_protocol": (
        "Participant interviews are audio-recorded with consent. "
        "Recordings are stored on an encrypted university server. "
        "No identifiable data leaves the approved computing environment. "
        "Data retention period is five years after study completion."
    ),
    "coding_manual": (
        "Indirect financial assistance includes informal loans from family or friends, "
        "community fund contributions, and employer advances. "
        "Code as INDIRECT_FIN when participant describes receiving money not from a formal institution. "
        "Do not code crowdfunding proceeds as INDIRECT_FIN unless the participant explicitly "
        "requested the funds for the study purpose."
    ),
    "missing_data_policy": (
        "Wave 2 missing data: use multiple imputation with five imputed datasets. "
        "Variables with more than 40 percent missingness are excluded from imputation "
        "and flagged for sensitivity analysis. "
        "Report both complete-case and imputed results in supplemental tables."
    ),
    "interview_protocol": (
        "Interviews are conducted in a private setting chosen by the participant. "
        "Duration is 60 to 90 minutes. "
        "The interviewer reviews the consent form verbally before recording begins. "
        "Participants may stop or pause the recording at any time without consequence."
    ),
}

print(f"Loaded {len(DOCUMENTS)} documents:")
for name in DOCUMENTS:
    print(f"  - {name}")


## Step 4 — Define the tool

We define a `search_documents` function that the model can call when it needs to look something up. We also write a schema that describes the function to the model in structured terms — this is how the model knows what the tool does and what arguments to pass.

In a real pipeline, this function would query a vector database with semantic search (as in the RAG chapter). Here we use simple keyword matching to keep the focus on the agent loop itself.

In [ ]:
def search_documents(query: str) -> str:
    """Search the document collection for passages relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Quick test
print(search_documents("audio recording consent"))


## Step 5 — Set up the Gemini model with function calling

We register the `search_documents` function as a tool that the model is allowed to call. Gemini's SDK lets you pass Python functions directly — it reads the docstring and type hints to understand what the function does.

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    tools=[search_documents],
    system_instruction=(
        "You are a helpful research assistant with access to a lab document collection. "
        "When you need specific information about protocols, coding rules, data handling, "
        "or interview procedures, use the search_documents tool. "
        "Only answer based on what you find in the documents. "
        "If the documents do not contain the answer, say so clearly."
    )
)

print("Model ready:", model.model_name)


## Step 6 — The agent loop

This is the core of the notebook. The loop works like this:

1. Send the model a question
2. If the model decides to call a tool, run the function and send the result back
3. The model reads the result and either calls another tool or writes a final answer
4. Repeat until done

Watch the printed output carefully — it shows you each decision the model makes and why.

In [ ]:
def run_agent(question: str) -> str:
    """Run a simple agent loop and return the final answer."""
    print(f"Question: {question}")
    print("-" * 50)

    chat = model.start_chat(enable_automatic_function_calling=False)
    response = chat.send_message(question)
    step = 1

    while True:
        # Check if the model wants to call a tool
        tool_calls = []
        for part in response.parts:
            if hasattr(part, "function_call") and part.function_call.name:
                tool_calls.append(part.function_call)

        if not tool_calls:
            # No tool calls — the model is done, extract the text answer
            for part in response.parts:
                if hasattr(part, "text") and part.text:
                    print(f"\nFinal answer:\n{part.text}")
                    return part.text
            return "(No text response)"

        # Execute each tool call and collect results
        tool_results = []
        for call in tool_calls:
            args = dict(call.args)
            print(f"Step {step}: model calls search_documents(query='{args.get('query', '')}')")
            result = search_documents(**args)
            print(f"  Result preview: {result[:100]}...")
            tool_results.append(
                genai.protos.Part(
                    function_response=genai.protos.FunctionResponse(
                        name=call.name,
                        response={"result": result}
                    )
                )
            )
            step += 1

        # Send the tool results back to the model
        response = chat.send_message(tool_results)

    return "(Loop ended unexpectedly)"


# Run a first example
_ = run_agent("Where are audio recordings stored, and for how long?")


## Step 7 — Try more questions

Run the agent on a few different questions. Notice how the model decides what to search for based on the question, not a fixed rule.

In [ ]:
questions = [
    "According to the coding manual, how should I code a loan from a participant's sibling?",
    "A variable in wave 2 has 45 percent missing values. What should I do?",
    "Can participants choose to be interviewed at a coffee shop?",
]

for q in questions:
    print("\n" + "=" * 60)
    run_agent(q)


## What you just saw

The key pattern to notice:

- The model **decided** to call the tool — you did not tell it to
- It **formulated the search query** based on the question — that is the planning step
- It **read the result** and used it to compose a grounded answer
- When the documents did not contain the answer, it said so rather than guessing

This is the agent loop in its simplest form. In a real research workflow the tools would be more powerful — a vector database over thousands of documents, code execution, file access — but the loop structure is identical.

**Context engineering, visible here:** Everything the model used to reason at each step was explicitly passed in the message history. The system prompt defined its role. The tool result was added to the conversation before the next response. Nothing is remembered automatically — it all lives in the context window.

## Next steps

- Replace `search_documents` with a real vector search using `sentence-transformers` (see Chapter 24)
- Try a two-tool setup — add a `summarize_findings` tool and watch the model coordinate between them
- Ask a question that requires two searches and see whether the model issues them sequentially
